# Stop In-Progress Fabric Workloads Across Capacities

This notebook finds workspaces on all dedicated capacities in a tenant and stops in-progress, starting, or unknown notebook/spark/pipeline executions.

It supports excluding one or more admin workspaces by display name.

In [ ]:
# =========================
# CONFIGURATION
# =========================

# Optional: set for logging/traceability only. Authentication comes from notebook identity.
TARGET_TENANT_ID = "<tenant-guid-or-name>"

# Workspace display names to skip entirely (case-insensitive).
EXCLUDED_WORKSPACE_NAMES = {
    "Admin",
    "Fabric Admin"
}

# Status values to stop (normalized to lowercase).
TARGET_STATUSES = {"inprogress", "starting", "unknown", "running"}

# Safety switch. Set to False to execute actual stop/cancel calls.
DRY_RUN = True

# Continuous monitoring loop configuration
LOOP_DURATION_MINUTES = 30
POLL_INTERVAL_SECONDS = 30

# API roots
FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"
PBI_ADMIN_BASE = "https://api.powerbi.com/v1.0/myorg/admin"

In [ ]:
import requests
import time
from typing import Dict, List, Iterable
from notebookutils import mssparkutils

fabric_token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")
pbi_token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")

fabric_headers = {
    "Authorization": f"Bearer {fabric_token}",
    "Content-Type": "application/json"
}

pbi_headers = {
    "Authorization": f"Bearer {pbi_token}",
    "Content-Type": "application/json"
}

def normalize_status(value: str) -> str:
    if value is None:
        return "unknown"
    return str(value).replace(" ", "").replace("_", "").lower()

def request_json(method: str, url: str, headers: Dict[str, str], body: Dict = None, retries: int = 5):
    for attempt in range(1, retries + 1):
        response = requests.request(method=method, url=url, headers=headers, json=body, timeout=60)
        if response.status_code == 429:
            wait_seconds = int(response.headers.get("Retry-After", "5"))
            print(f"  Throttled (429): waiting {wait_seconds}s, attempt {attempt}/{retries}")
            time.sleep(wait_seconds)
            continue

        if response.status_code >= 400:
            return None, response

        if response.text and response.text.strip():
            return response.json(), response

        return {}, response

    return None, response

def get_all_pages(start_url: str, headers: Dict[str, str]) -> List[Dict]:
    items: List[Dict] = []
    next_url = start_url

    while next_url:
        data, resp = request_json("GET", next_url, headers)
        if resp is None or resp.status_code >= 400 or data is None:
            code = resp.status_code if resp is not None else "N/A"
            message = resp.text[:400] if resp is not None else "No response"
            raise RuntimeError(f"Failed GET {next_url}. status={code}. body={message}")

        items.extend(data.get("value", []))
        next_url = data.get("continuationUri")

    return items

def safe_post(url: str, headers: Dict[str, str], body: Dict = None) -> bool:
    if DRY_RUN:
        print(f"  [DRY_RUN] POST {url}")
        return True

    _, resp = request_json("POST", url, headers, body=body)
    if resp is None:
        print(f"  Failed POST {url}: no response")
        return False

    if resp.status_code in (200, 202, 204):
        return True

    print(f"  Failed POST {url}: {resp.status_code} {resp.text[:300]}")
    return False

In [ ]:
# =========================
# DISCOVER WORKSPACES ON CAPACITIES
# =========================

# Requires Fabric/Power BI admin privileges.
admin_groups_url = f"{PBI_ADMIN_BASE}/groups?$top=5000"
all_groups = get_all_pages(admin_groups_url, pbi_headers)

excluded_lower = {x.strip().lower() for x in EXCLUDED_WORKSPACE_NAMES}

target_workspaces: List[Dict] = []
for g in all_groups:
    name = g.get("name") or g.get("displayName") or ""
    ws_id = g.get("id")
    cap_id = g.get("capacityId")

    # Only dedicated capacity workspaces are included.
    if not cap_id:
        continue

    if name.strip().lower() in excluded_lower:
        print(f"Skipping excluded workspace: {name} ({ws_id})")
        continue

    target_workspaces.append({
        "id": ws_id,
        "displayName": name,
        "capacityId": cap_id
    })

capacity_ids = sorted({w["capacityId"] for w in target_workspaces})
print(f"Tenant hint: {TARGET_TENANT_ID}")
print(f"Workspaces on capacity to process: {len(target_workspaces)}")
print(f"Capacities in scope: {len(capacity_ids)}")

In [ ]:
# =========================
# STOP NOTEBOOK SPARK SESSIONS
# =========================

summary = {
    "workspaces_scanned": 0,
    "notebook_sessions_seen": 0,
    "notebook_sessions_stopped": 0,
    "item_jobs_seen": 0,
    "item_jobs_cancelled": 0,
    "errors": 0
}

def list_workspace_notebooks(workspace_id: str) -> List[Dict]:
    url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/notebooks"
    try:
        return get_all_pages(url, fabric_headers)
    except Exception as ex:
        print(f"  Failed listing notebooks for workspace {workspace_id}: {ex}")
        summary["errors"] += 1
        return []

def stop_notebook_sessions(workspace_id: str, notebook: Dict):
    notebook_id = notebook.get("id")
    notebook_name = notebook.get("displayName", notebook_id)

    sessions_url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/notebooks/{notebook_id}/sessions"
    data, resp = request_json("GET", sessions_url, fabric_headers)

    if resp is None or resp.status_code >= 400 or data is None:
        code = resp.status_code if resp is not None else "N/A"
        print(f"    Could not list sessions for notebook {notebook_name}. status={code}")
        summary["errors"] += 1
        return

    for session in data.get("value", []):
        state = normalize_status(session.get("state"))
        summary["notebook_sessions_seen"] += 1

        if state in TARGET_STATUSES:
            session_id = session.get("id")
            stop_url = f"{sessions_url}/{session_id}/stop"
            print(f"    Stop notebook session: {notebook_name} | state={state} | sessionId={session_id}")
            ok = safe_post(stop_url, fabric_headers)
            if ok:
                summary["notebook_sessions_stopped"] += 1

In [ ]:
# =========================
# CANCEL NOTEBOOK/PIPELINE JOB INSTANCES (CONTINUOUS LOOP)
# =========================

CANDIDATE_ITEM_TYPES = {"notebook", "pipeline", "datapipeline"}

def list_workspace_items(workspace_id: str) -> List[Dict]:
    url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items"
    try:
        return get_all_pages(url, fabric_headers)
    except Exception as ex:
        print(f"  Failed listing items for workspace {workspace_id}: {ex}")
        summary["errors"] += 1
        return []

def iter_job_instances(workspace_id: str, item_id: str) -> Iterable[Dict]:
    urls = [
        f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items/{item_id}/jobs/instances",
        f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items/{item_id}/jobInstances"
    ]

    for url in urls:
        data, resp = request_json("GET", url, fabric_headers)
        if resp is not None and resp.status_code < 400 and data is not None:
            for instance in data.get("value", []):
                yield instance
            return

def cancel_job_instance(workspace_id: str, item_id: str, instance_id: str) -> bool:
    urls = [
        f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items/{item_id}/jobs/instances/{instance_id}/cancel",
        f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items/{item_id}/jobInstances/{instance_id}/cancel"
    ]

    for url in urls:
        ok = safe_post(url, fabric_headers)
        if ok:
            return True

    return False

LOOP_DURATION_SECONDS = LOOP_DURATION_MINUTES * 60
loop_started_at = time.time()
loop_ends_at = loop_started_at + LOOP_DURATION_SECONDS
iteration = 0

print(f"Starting continuous stop loop for {LOOP_DURATION_MINUTES} minute(s). Poll interval: {POLL_INTERVAL_SECONDS}s")

while time.time() < loop_ends_at:
    iteration += 1
    now = time.time()
    remaining_seconds = max(0, int(loop_ends_at - now))

    iteration_summary = {
        "workspaces_scanned": 0,
        "notebook_sessions_seen": 0,
        "notebook_sessions_stopped": 0,
        "item_jobs_seen": 0,
        "item_jobs_cancelled": 0,
        "errors": 0
    }

    print("\n" + "=" * 100)
    print(f"Loop iteration {iteration} | Remaining: {remaining_seconds}s")

    for ws in target_workspaces:
        ws_id = ws["id"]
        ws_name = ws["displayName"]
        summary["workspaces_scanned"] += 1
        iteration_summary["workspaces_scanned"] += 1

        print("\n" + "-" * 80)
        print(f"Workspace: {ws_name} ({ws_id}) | Capacity: {ws['capacityId']}")

        notebooks = list_workspace_notebooks(ws_id)
        before_sessions_seen = summary["notebook_sessions_seen"]
        before_sessions_stopped = summary["notebook_sessions_stopped"]
        before_errors = summary["errors"]

        for nb in notebooks:
            stop_notebook_sessions(ws_id, nb)

        iteration_summary["notebook_sessions_seen"] += summary["notebook_sessions_seen"] - before_sessions_seen
        iteration_summary["notebook_sessions_stopped"] += summary["notebook_sessions_stopped"] - before_sessions_stopped
        iteration_summary["errors"] += summary["errors"] - before_errors

        items = list_workspace_items(ws_id)
        for item in items:
            item_type = str(item.get("type", "")).lower()
            if item_type not in CANDIDATE_ITEM_TYPES:
                continue

            item_id = item.get("id")
            item_name = item.get("displayName", item_id)

            for run in iter_job_instances(ws_id, item_id):
                status = normalize_status(run.get("status") or run.get("state"))
                summary["item_jobs_seen"] += 1
                iteration_summary["item_jobs_seen"] += 1

                if status in TARGET_STATUSES:
                    instance_id = run.get("id")
                    print(f"    Cancel item run: {item_name} [{item_type}] | status={status} | runId={instance_id}")
                    if cancel_job_instance(ws_id, item_id, instance_id):
                        summary["item_jobs_cancelled"] += 1
                        iteration_summary["item_jobs_cancelled"] += 1

    print("\nIteration Summary")
    for k, v in iteration_summary.items():
        print(f"{k}: {v}")

    if time.time() >= loop_ends_at:
        break

    sleep_seconds = min(POLL_INTERVAL_SECONDS, max(0, int(loop_ends_at - time.time())))
    if sleep_seconds > 0:
        print(f"Sleeping {sleep_seconds}s before next scan...")
        time.sleep(sleep_seconds)

print("\nContinuous loop window completed.")

In [ ]:
total_elapsed_seconds = int(time.time() - loop_started_at)

print("\n" + "#" * 100)
print("Execution Summary")
print(f"Loop duration target (minutes): {LOOP_DURATION_MINUTES}")
print(f"Loop elapsed (seconds): {total_elapsed_seconds}")
for k, v in summary.items():
    print(f"{k}: {v}")

if DRY_RUN:
    print("\nDRY_RUN is enabled. No stop/cancel action was executed.")
else:
    print("\nStop/cancel actions were executed.")